In [ ]:
import numpy as np
import librosa
import scipy.ndimage
import IPython.display as ipd
import matplotlib.pyplot as plt

# 1. Load the audio file (replace with your file path)
# We load it in mono for simplicity in this demo
audio_path = "songs/test.mp3"
x, sr = librosa.load(audio_path, sr=None, mono=True)

print(f"Audio loaded! Duration: {len(x)/sr:.2f} seconds")

In [ ]:
# Parameters for STFT
n_fft = 2048
hop_length = 512

# Compute STFT (Complex matrix)
X = librosa.stft(x, n_fft=n_fft, hop_length=hop_length)

# Compute Power Spectrogram (Real, positive matrix)
# librosa STFT shape is (frequencies, time_frames)
Y = np.abs(X)**2

In [ ]:
# Filter lengths (L_h and L_p must be odd numbers)
L_h = 31  # Horizontal (Time) filter length
L_p = 31  # Vertical (Frequency) filter length

# Apply median filters using scipy's ndimage
# size=(1, L_h) slides horizontally across time frames
Y_tilde_h = scipy.ndimage.median_filter(Y, size=(1, L_h))

# size=(L_p, 1) slides vertically across frequency bins
Y_tilde_p = scipy.ndimage.median_filter(Y, size=(L_p, 1))

In [ ]:
epsilon = 1e-10

# Calculate Soft Masks based on the LaTeX formulas
M_h = (Y_tilde_h + (epsilon / 2)) / (Y_tilde_h + Y_tilde_p + epsilon)
M_p = (Y_tilde_p + (epsilon / 2)) / (Y_tilde_h + Y_tilde_p + epsilon)

# Apply masks to the ORIGINAL COMPLEX STFT
X_h = M_h * X
X_p = M_p * X

In [ ]:
# Inverse STFT to get the audio arrays
x_h = librosa.istft(X_h, hop_length=hop_length, length=len(x))
x_p = librosa.istft(X_p, hop_length=hop_length, length=len(x))

print("Separation Complete!")

In [ ]:
# Play the results
print("Original Audio:")
display(ipd.Audio(x, rate=sr))

print("Harmonic Component (x^h):")
display(ipd.Audio(x_h, rate=sr))

print("Percussive Component (x^p):")
display(ipd.Audio(x_p, rate=sr))

# IMPORTANT: Remember to "Clear All Outputs" before committing to Git! ;)

In [ ]:
# Calcoliamo gli spettrogrammi di potenza EFFETTIVI (mascherati) usati per la ricostruzione
# NB: Calcoliamo il modulo quadro dei segnali complessi mascherati X_h e X_p
Y_h_final = np.abs(X_h)**2
Y_p_final = np.abs(X_p)**2

# Conversione in Decibel (dB)
# Usiamo ref=np.max per normalizzare lo 0 dB al valore massimo dello spettrogramma originale
# Questo garantisce che la scala dei colori sia coerente tra i tre grafici.
D_orig = librosa.power_to_db(Y, ref=np.max)
D_h = librosa.power_to_db(Y_h_final, ref=np.max)
D_p = librosa.power_to_db(Y_p_final, ref=np.max)

print("Spettrogrammi convertiti in dB. Pronti per il plotting.")

In [ ]:
# Creiamo la figura (utilizziamo figsize=(18, 5) per tre colonne larghe e basse)
# sharex=True e sharey=True allineano gli assi
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(18, 5), sharex=True, sharey=True)

# Parametri comuni per la visualizzazione
# 'y_axis="log"' mostra la frequenza in scala logaritmica, utile per la musica
specshow_kwargs = dict(sr=sr, hop_length=hop_length, x_axis='time', y_axis='log', cmap='magma')

# --- 1. Spettrogramma Originale ---
img1 = librosa.display.specshow(D_orig, ax=ax[0], **specshow_kwargs)
ax[0].set_title('Spettrogramma Originale (x)', fontsize=14)
# Rimuoviamo l'etichetta dell'asse Y per le colonne successive per pulizia
ax[0].set_ylabel('Frequenza (Hz) [Log]', fontsize=12)

# --- 2. Componente Armonica ---
img2 = librosa.display.specshow(D_h, ax=ax[1], **specshow_kwargs)
ax[1].set_title('Componente Armonica (x^h)', fontsize=14)
# Rimuoviamo l'etichetta dell'asse Y (è condivisa)
ax[1].set_ylabel('') 

# --- 3. Componente Percussiva ---
img3 = librosa.display.specshow(D_p, ax=ax[2], **specshow_kwargs)
ax[2].set_title('Componente Percussiva (x^p)', fontsize=14)
ax[2].set_ylabel('')

# Aggiungiamo UNA singola colorbar a destra (poiché condividono la stessa scala dB)
fig.colorbar(img1, ax=ax, format="%+2.0f dB")

# Etichetta asse X comune
for a in ax:
    a.set_xlabel('Tempo (s)', fontsize=12)

# Regoliamo il layout automaticamente per evitare sovrapposizioni
plt.tight_layout()

# --- ESPORTAZIONE PER LA TESI ---
# Rimuovi il commento dalla riga seguente per salvare l'immagine in alta risoluzione (300 dpi)
# plt.savefig('HPS_separazione_spettrogrammi.png', dpi=300, bbox_inches='tight')

plt.show()